In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EMP05 - Contingent Workers in High-Risk Jurisdictions
   
   BUSINESS QUESTION: 
   Are there contingent workers associated with the Assessable Unit operating in 
   high-risk jurisdictions?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGH-RISK JURISDICTIONS: Queries the td_country_risk_rating_abac table to isolate 
      countries where the FinalABACRating is strictly 'High'.
   3. CONTINGENT WORKERS: Filters the HR Enterprise List to find employees where 
      WorkerType contains the word 'contingent'. Cleans the CostCenterID using 
      TRY_CAST to integer to prevent leading-zero mismatch issues.
   4. FILTERING & MAPPING: INNER JOINS the contingent workers to the High-Risk table 
      to strictly keep high-risk workers. Then maps them to AU_IDs via the CC Bootstrap.
   5. OUTPUT FORMATTING: Rolls up the data by AU_ID. LEFT JOINS this to the Master AU 
      anchor. If mapped high-risk workers exist, outputs 'Yes'. Otherwise, 'No'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    -- 2. Grab jurisdictions strictly rated as 'High' risk
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Contingent_Workers AS (
    -- 3. Filter HR List for Contingent Workers and prep the Cost Center for joining
    SELECT 
        TRY_CAST(TRIM(CostCenterID) AS INT) AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry,
        TRIM(WorkerType) AS WorkerType
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

High_Risk_Contingent_Workers AS (
    -- Strictly keep only the contingent workers in high-risk countries
    SELECT 
        c.Cleaned_CC,
        c.Workcountry
    FROM Contingent_Workers c
    INNER JOIN High_Risk_Countries h
        ON c.Workcountry = h.CountryName
    WHERE c.Cleaned_CC IS NOT NULL
),

CC_Mapping AS (
    -- Standardize the CC Mapping table
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    -- 4. Map the high-risk contingent workers to their respective AU_IDs
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM High_Risk_Contingent_Workers hw
    INNER JOIN CC_Mapping m
        ON hw.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 5. Final Output: Strictly structured to the requested 6 columns anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EMP05' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EMP05 Drill-Down
   
   PURPOSE: Shows EVERY high-risk contingent worker found in the HR data, regardless 
   of whether their Cost Center maps to an AU, or whether that AU exists in the Master 
   List. Useful for tracking unmapped high-risk contingent headcount.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Contingent_Workers AS (
    SELECT 
        CostCenterID AS Raw_String,
        TRY_CAST(TRIM(CostCenterID) AS INT) AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry,
        TRIM(WorkerType) AS WorkerType
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

High_Risk_Contingent_Workers AS (
    SELECT 
        c.Raw_String,
        c.Cleaned_CC,
        c.Workcountry,
        c.WorkerType
    FROM Contingent_Workers c
    INNER JOIN High_Risk_Countries h
        ON c.Workcountry = h.CountryName
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    COALESCE(m.AU_ID, 'UNMAPPED_CC') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    hw.Cleaned_CC AS Flagged_Cost_Center,
    hw.Workcountry AS High_Risk_Jurisdiction,
    hw.WorkerType,
    hw.Raw_String AS Original_Cost_Center_Data
FROM High_Risk_Contingent_Workers hw
LEFT JOIN CC_Mapping m
    ON hw.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast
    ON m.AU_ID = mast.BusinessID
ORDER BY BusinessID, hw.Cleaned_CC;